# 4. Distributions — a hypothesis about the process

A distribution is a hypothesis about the process that generated the data.

"Lognormal with μ=3.2, σ=0.8" is a description you can check against what you know about
how the numbers were produced: a lognormal says the value is a *product* of many factors,
which is what gives it the long tail. "Mean 34, std 41" describes nothing, because for a
lognormal the mean is not typical and the std implies a symmetry that is not there.

Three reasons to fit a distribution rather than report a mean, in increasing order of value:

1. **To summarise honestly.** Two parameters that describe the shape, instead of two that
   assume one.
2. **To decide what is unusual.** A point is only an outlier *relative to a distribution*.
   "Three standard deviations from the mean" is a statement about the normal distribution
   you did not know you had assumed.
3. **To compare two situations.** If the same family fits before and after an event, the
   parameters say *what* changed — the rate, the spread, the scale — and not just *that*
   something did. That is the move this notebook builds towards.

## What goad gives you for this

Lesson 1 gave data decisions a home in a `Pipeline`, lesson 2 gave plots one in `BasePlot`.
Distribution fitting gets the same treatment, in three pieces:

- **`DistributionRegistry`** holds the families goad will try — `norm`, `lognorm`,
  `exponential`, `gamma`, `weibull`, `poisson`, `nbinom`, and a few more — each with the
  metadata a fit needs: the scipy object, whether it is discrete, how many parameters. You
  can register a family it does not ship (`registry.register_distribution("pareto",
  stats.pareto, is_discrete=False, num_params=3)`), and every registry starts from the
  shipped set, so a registration in one cell does not leak into the next.
- **`DistributionFitter(registry, seed=42).fit(data, discrete=...)`** fits every family
  of the right kind by maximum likelihood and returns one result per family — a `FitResult` with
  the parameters, a `frozen_dist` you can draw or sample from, the log-likelihood, and a
  Kolmogorov–Smirnov test; or a `FailedFit` carrying the reason. Failures are values, so
  one badly-behaved family does not lose you the comparison. `discrete` is *your* call:
  counts are discrete, durations are not, and getting it wrong does not error — it fits
  the wrong half of the registry. `fit_distribution("lognorm", data)` fits one family
  when you already know which. The `seed` is there because the optimizer behind the fit
  is stochastic: pass one when the numbers must repeat, as they must in a lesson; leave it
  out in your own work and a fit that moves between runs is telling you something.
- **`fit_table(results)`** turns those results into the ranked table you hand in: family,
  parameters, log-likelihood, KS statistic and p-value, and which criteria it won.

Two winners are marked, on purpose. **Log-likelihood** rewards a family that puts high
probability on the points you observed — it is dominated by the bulk of the data. **KS**
measures the largest gap between the fitted and the empirical cumulative distribution — it
is sensitive to the middle and comparatively blind in the tails. They usually agree; when
they do not, that disagreement is the most informative thing on the table, and the answer
is a picture, not an average of the two numbers.

The pictures: `PlotFits` (the histogram with the top fits over it), `QQPlot` (sorted data
against a fitted family's quantiles — the tails, where families differ and histograms are
unreadable), `ECDFPlot` (two samples on one bin-free axis), and `DistPlot` (draw a family
on its own, or over a histogram). All of them are `BasePlot`s: `create_figure(n_plots=...)`
and `plot_on_axes(...)` compose them the way every plot in this course composes.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from goad_toolkit.analytics import DistributionFitter, FitResult, NullDistribution, fit_table
from goad_toolkit.datatransforms import Filter, GroupAgg, Pipeline
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import (
    DistPlot,
    ECDFPlot,
    FitPlotSettings,
    HistogramPlot,
    NullPlot,
    PlotFits,
    PlotSettings,
    QQPlot,
)

from wa_analyzer.data import load_own_chat, load_showcase

rng = np.random.default_rng(42)

## 4.1 Six families, generated — before any real data is involved

Each family below is sampled from a known generator and drawn with its own curve over the
sample, so what you see is *only* the shape: no fitting, no "did we estimate the parameters
right". The one-liner under each name is the mechanism that produces it — the thing to
recognise in your own data before you fit anything.

In [ ]:
families = [
    ("norm", stats.norm(loc=0, scale=1), "sums of many independent contributions"),
    ("lognorm", stats.lognorm(s=0.8, scale=np.exp(1)), "products of many factors"),
    ("expon", stats.expon(scale=1), "time between events arriving at a steady rate"),
    ("poisson", stats.poisson(mu=4), "counts of events in a fixed window, steady rate"),
    ("weibull", stats.weibull_min(c=1.5, scale=1), "time-to-failure with a changing hazard"),
    ("pareto", stats.pareto(b=2.5), "preferential attachment, rich-get-richer"),
]

gallery = PlotSettings(
    figsize=(13, 7),
    title="Six families, generated",
    subplot_titles=[f"{name}\n{mechanism}" for name, _, mechanism in families],
    max_cols=3,
)
host = HistogramPlot(gallery)
fig, axes = host.create_figure(n_plots=len(families))
for ax, (name, dist, mechanism) in zip(axes, families):
    sample = dist.rvs(2000, random_state=rng)
    # discrete=True gives a count one bar per integer, so the bars and the pmf share a scale
    host.plot_on_axes(HistogramPlot(gallery), ax, data=sample, color="lightgrey",
                      discrete=(name == "poisson"))
    host.plot_on_axes(DistPlot(gallery), ax, distribution=dist, color="crimson")
    ax.set_ylabel("")

Two of these are counts, not measurements: `poisson` has mass only at the integers, so its
curve is drawn through them — a discrete count has no density, only a probability per value.
And two of them are hard to tell apart from a histogram: `lognorm` and `pareto` both just
look "long-tailed". The discriminator is the rank–frequency plot on log-log axes (a power law
is straight there, a lognormal curves) — goad's Distributions chapter, §5.7, if you ever
need it.

## 4.2 The central limit theorem — sums converge, nothing else has to

Sums of *anything* with finite variance drift toward normal as you add more terms. That is
why normal turns up wherever sums turn up — and it says nothing at all about products,
waiting times, or counts, which converge to something else entirely (lognormal, gamma,
Poisson) precisely because they are not sums.

In [ ]:
ns = [1, 2, 5, 30]
clt = PlotSettings(
    figsize=(14, 3.2),  # ty: ignore[invalid-argument-type]
    title="Sums of uniforms converge to normal as n grows",
    subplot_titles=[f"sum of {n} uniform draw{'s' if n > 1 else ''}" for n in ns],
    max_cols=4,
    sharex=True,
    sharey=True,
)
host = HistogramPlot(clt)
fig, axes = host.create_figure(n_plots=len(ns))
for ax, n in zip(axes, ns):
    sums = stats.uniform(loc=-1, scale=2).rvs((5000, n), random_state=rng).sum(axis=1)
    sums = (sums - sums.mean()) / sums.std()
    host.plot_on_axes(HistogramPlot(clt), ax, data=sums, color="steelblue")
    host.plot_on_axes(DistPlot(clt), ax, distribution=stats.norm(0, 1), x_range=(-4, 4))
    ax.set_ylabel("")

In [ ]:
# A PRODUCT of the same uniforms does not converge to normal at all: it converges to
# lognormal, because a sum of logs is a sum, and exp() of a sum is a product.
n = 30
sums = stats.uniform(loc=1, scale=1).rvs((5000, n), random_state=rng).sum(axis=1)
products = stats.uniform(loc=1, scale=1).rvs((5000, n), random_state=rng).prod(axis=1)

product = PlotSettings(
    figsize=(12, 3.2),  # ty: ignore[invalid-argument-type]
    title="The CLT is a statement about sums",
    subplot_titles=["sum → normal", "product → long tail", "log(product) → normal again"],
    max_cols=3,
)
host = HistogramPlot(product)
fig, axes = host.create_figure(n_plots=3)
for ax, values in zip(axes, [sums, products, np.log(products)]):
    host.plot_on_axes(HistogramPlot(product), ax, data=values, color="steelblue")
    ax.set_ylabel("")

The right panel is `log(product)`, not `product` — that is the whole trick lognormal is
named for. Plot `products` directly and it is skewed and long-tailed, not bell-shaped,
because a product of positive numbers is exactly the kind of variable the CLT does not
apply to directly. Take the log and it becomes a sum, and the theorem applies again.

## 4.3 The long tail, on taxi fares

`taxis`: 6,433 NYC rides. The question is not "what does a fare look like" — it is *which
family, and why*, because the answer decides what counts as an expensive ride.

Think about the mechanism first. A fare is a base charge plus a meter that runs on distance
*and* on time stuck in traffic, plus tolls and surcharges on some rides. Distance itself is
skewed — most rides are short, a few cross the city — and slow traffic multiplies the cost
of every extra kilometre. Several factors, mostly *multiplying* each other, with a hard
floor at the base charge and no ceiling: that is the product mechanism from §4.2, and it
predicts a lognormal. Check the prediction in three steps: the summary that betrays the
tail, the picture with and without the log, and then the fitter.

In [ ]:
taxis = load_showcase("taxis")
fare = taxis["fare"].to_numpy()

print(f"mean:   {fare.mean():.2f}")
print(f"median: {np.median(fare):.2f}")

The mean sits well above the median — the tail pulls it up. For a symmetric distribution the
two would match, so this gap is the first sign that "mean ± std" would describe a fare that
does not exist. Now the picture, twice: raw, and after taking the log. **The log is the
fix**: if the mechanism is multiplicative, `log(fare)` is a *sum* of the logged factors, and
§4.2 says a sum should look normal.

In [ ]:
tail = PlotSettings(
    figsize=(10, 3.5),  # ty: ignore[invalid-argument-type]
    title="The same fares, before and after the log",
    subplot_titles=["fare: a long right tail", "log(fare): the tail becomes a bell"],
    max_cols=2,
)
host = HistogramPlot(tail)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(HistogramPlot(tail), axes[0], data=fare, color="steelblue")
_ = host.plot_on_axes(HistogramPlot(tail), axes[1], data=np.log(fare), color="steelblue")

### Let the fitter say it

The picture supports the prediction; the fitter makes it a number. `fit()` on the raw fares
tries every continuous family in the registry and ranks them. Read the table for two things:
does `lognorm` win, and by how much does it beat `norm` — the family a z-score rule quietly
assumes.

In [ ]:
fitter = DistributionFitter(DistributionRegistry(), seed=42)
fare_fits = fitter.fit(fare, discrete=False)
fit_table(fare_fits)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

In [ ]:
fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="fare", ylabel="density", title="Taxi fares: the top three fits")).plot(
    data=fare, fit_results=fare_fits, fitplotsettings=FitPlotSettings(bins=40, max_fits=3),
)

A histogram with a curve on it is dominated by the bulk of the data, which is exactly where
families look most alike. The tail is where they differ, and the tail is what a `QQPlot`
shows: sorted fares against the quantiles a fitted family predicts, with a reference line.
Points that bend away from the line are values the family cannot explain.

In [ ]:
norm_fit = fitter.fit_distribution("norm", fare)
lognorm_fit = fitter.fit_distribution("lognorm", fare)

qq = PlotSettings(
    figsize=(10, 4.5),  # ty: ignore[invalid-argument-type]
    title="Which family explains the expensive rides?",
    subplot_titles=["fare vs. fitted normal", "fare vs. fitted lognormal"],
    xlabel="theoretical quantile",
    ylabel="fare",
    max_cols=2,
)
host = QQPlot(qq)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(QQPlot(qq), axes[0], data=fare, distribution=norm_fit.frozen_dist)  # ty: ignore[unresolved-attribute]
_ = host.plot_on_axes(QQPlot(qq), axes[1], data=fare, distribution=lognorm_fit.frozen_dist)  # ty: ignore[unresolved-attribute]

The normal qq-plot bends sharply away from the line in the upper tail — exactly where a
histogram is least readable, and exactly where a "z-score above 3" rule would flag ordinary
expensive rides as anomalies. The lognormal tracks the line much further out. Same data;
the difference is what each family expects the tail to look like.

What that means for the single most expensive ride in the set:

In [ ]:
worst_fare = fare.max()
z = (worst_fare - fare.mean()) / fare.std()
tail_prob = 1 - lognorm_fit.frozen_dist.cdf(worst_fare)  # ty: ignore[unresolved-attribute]
print(f"most expensive fare: ${worst_fare:.2f}, n={len(fare):,} rides")
print(f"z-score under a normal fit: {z:.1f}  (a real normal puts this at ~1-in-10^32)")
print(f"P(fare >= this) under the fitted lognormal: {tail_prob:.1e}  (~1-in-{1/tail_prob:,.0f})")

Still rare under the lognormal fit — 1-in-19,000 is not "expected" — but the two models
disagree by about 27 orders of magnitude on *how* rare. The normal model calls this ride
essentially impossible; the lognormal model calls it an unlucky but real draw from a tail
every long ride is exposed to. Neither model says "delete this row"; only the wrong one says
"this cannot have happened." That is reason 2 from the top of the notebook: an outlier is a
claim about a distribution, and the claim is only as good as the family behind it.

## 4.4 A count, before and after an event

Everything so far described one situation. The move that turns a fit into evidence is to fit
the same family on two sides of an event you already know about, and read what changed. The
event here is the one 03.1 used: Ubuntu releases, ten Thursdays between 2013 and 2017, in
the two channels of the `ubuntu_irc_release_hourly` showcase — `#ubuntu` (the big one) and
`#ubuntu-it`. The variable is the simplest count there is: **messages per Thursday**. The
showcase arrives as hourly totals with the release days already flagged, so the counting is
one `GroupAgg` per channel.

In [ ]:
hourly = load_showcase("ubuntu_irc_release_hourly")


def messages_per_day(channel: str) -> pd.DataFrame:
    """One row per Thursday: total messages that day, and whether it was a release day."""
    return (
        Pipeline()
        .add(Filter, expr=f"channel == '{channel}'")
        .add(GroupAgg, by=["date", "is_release"], column="messages", agg="sum")
        .apply(hourly)
    )


it_days = messages_per_day("#ubuntu-it")
it_ordinary = it_days.loc[~it_days.is_release, "messages"].to_numpy().astype(float)
it_release = it_days.loc[it_days.is_release, "messages"].to_numpy().astype(float)
print(f"#ubuntu-it: {len(it_ordinary)} ordinary Thursdays, {len(it_release)} release days")
print(f"ordinary: mean {it_ordinary.mean():.0f}, variance {it_ordinary.var():.0f}")

### Step one: which family, on an ordinary Thursday?

Messages per day is a count, so the textbook hypothesis is **Poisson**: independent events
arriving at a steady rate. Poisson makes a checkable promise — its variance equals its mean.
The printout above already breaks that promise by a factor of a few hundred, so the fit
should fail, and *how* it fails is the lesson. Fit every discrete family and draw the two
that matter over the data.

In [ ]:
it_fits = fitter.fit(it_ordinary, discrete=True)
fit_table(it_fits)[["distribution", "params", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

In [ ]:
by_name = {f.distribution: f for f in it_fits if isinstance(f, FitResult)}

steady = PlotSettings(
    figsize=(10, 4),
    title="#ubuntu-it, ordinary Thursdays: a steady rate is the wrong hypothesis",
    xlabel="messages per day",
    ylabel="density",
)
host = HistogramPlot(steady)
fig, ax = host.plot(data=it_ordinary, bins=30, color="lightgrey")
host.plot_on(DistPlot(steady), distribution=by_name["poisson"].frozen_dist,
             color="crimson", label="poisson: one steady rate")
_ = host.plot_on(DistPlot(steady), distribution=by_name["nbinom"].frozen_dist,
                 x_range=(0, it_ordinary.max()), color="steelblue", label="nbinom: a rate that wanders")

The Poisson is a needle. With a mean of roughly 560 it allows day-to-day swings of about ±25
messages; the channel swings between under 100 and over 1,500. Its KS p-value in the table is
zero to fifty decimal places, and that rejection is not a failure of the fit — it is the
finding: **there is no steady rate.** Which Thursday it is matters: the year (the channel
shrank across the window), who happened to be around, what broke that week.

The **negative binomial** is the family for exactly that: a Poisson whose rate is itself
drawn at random each day. Its two parameters are a mean and a dispersion — how far the rate
wanders — and it is the top row of the table, with a KS p-value that does not reject it. So
the honest summary of an ordinary Thursday is not "about 560 messages" but "a rate of about
560 that routinely halves or triples". That is reason 1 again: a description that fits.

### Step two: the same family, on release days

Ten release days is a small sample, but the question is small too: does the family hold, and
which parameter moved? Fit it, and put both samples on one bin-free axis.

In [ ]:
release_fits = fitter.fit(it_release, discrete=True)
release_best = next(f for f in release_fits if isinstance(f, FitResult) and f.best_likelihood)
ordinary_best = by_name["nbinom"]

print(f"ordinary Thursday: {ordinary_best.distribution}, mean {ordinary_best.frozen_dist.mean():.0f}, "
      f"sd {ordinary_best.frozen_dist.std():.0f}")
print(f"release day:       {release_best.distribution}, mean {release_best.frozen_dist.mean():.0f}, "
      f"sd {release_best.frozen_dist.std():.0f}")
print(f"rate on a release day: x{it_release.mean() / it_ordinary.mean():.2f}")

fig, ax = ECDFPlot(PlotSettings(figsize=(8, 4), title="#ubuntu-it: ordinary Thursdays vs release days",
                      xlabel="messages per day", ylabel="share of days at or below")).plot(
    data=it_ordinary, compare=it_release, label="ordinary Thursday", compare_label="release day",
)

Same family on both sides, and what moved is the **rate**: a release day runs at roughly
1.4× an ordinary Thursday, with a similar spread around it. The ECDF shows the same thing
without a single bin: the red curve sits to the right of the blue one along most of its
length — release days are shifted, not reshaped. That sentence, *the rate moved and the
shape did not*, is what fitting on both sides buys over comparing two averages.

### Step three: how sure?

A 1.4× lift on ten days, drawn from a distribution that "routinely halves or triples", could
be luck. The check is the one this course keeps coming back to: **what would this number look
like if the label meant nothing?** `NullDistribution` shuffles the `is_release` flag among
the Thursdays, recomputes the rate difference, and does that a thousand times. `NullPlot`
draws the cloud and marks the real value in it.

In [ ]:
def rate_difference(days: pd.DataFrame) -> float:
    """Mean messages on release days minus mean on ordinary Thursdays."""
    means = days.groupby("is_release").messages.mean()
    return means[True] - means[False]


null = NullDistribution(rate_difference, n_iter=2000, seed=4)
it_null = null.run(it_days, label="is_release")

fig, ax = NullPlot(PlotSettings(figsize=(8, 4), title="#ubuntu-it: is a +230 lift more than chance?",
                                xlabel="release − ordinary, messages per day")).plot(result=it_null)
ax.legend()
print(f"two-sided p: {it_null.p_value():.3f}   one-sided p (release days busier): {it_null.p_value('greater'):.3f}")

The observed lift sits in the upper tail of the cloud but not outside it — a two-sided
p-value around 0.07 (one-sided, "busier", about 0.04). Ten days of a rate that wanders this
much is a thin basis, and the number says so.

Now the same three steps on `#ubuntu`, the channel with ten times the traffic.

In [ ]:
ubuntu_days = messages_per_day("#ubuntu")
ubuntu_null = null.run(ubuntu_days, label="is_release")

for name, days, result in [("#ubuntu-it", it_days, it_null), ("#ubuntu", ubuntu_days, ubuntu_null)]:
    ordinary = days.loc[~days.is_release, "messages"]
    lift = days.loc[days.is_release, "messages"].mean() / ordinary.mean()
    print(f"{name:11s} lift x{lift:.2f}   ordinary-day spread sd/mean = {ordinary.std() / ordinary.mean():.2f}"
          f"   two-sided p = {result.p_value():.4f}")

compare = PlotSettings(
    figsize=(13, 4),
    title="The same lift, two verdicts",
    subplot_titles=["#ubuntu-it: rate x1.4, borderline", "#ubuntu: rate x1.5, unmistakable"],
    xlabel="release − ordinary, messages per day",
    ylabel="density",
    max_cols=2,
)
host = NullPlot(compare)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(NullPlot(compare), axes[0], result=it_null)
host.plot_on_axes(NullPlot(compare), axes[1], result=ubuntu_null)
for ax in axes:
    ax.legend()

Both channels show a release lift of about one and a half. In `#ubuntu` it is unmistakable:
the real value stands far outside anything the shuffled labels produce. In `#ubuntu-it` it is
borderline. **The effect is the same size; the evidence is not**, and the printout says why:
an ordinary Thursday in `#ubuntu` wanders about a third of its mean, in `#ubuntu-it` about
two thirds. Many independent people averaging out make days more alike — §4.2 again, sums of
many contributions — so the same lift stands clear of the day-to-day noise in the big channel
and drowns in it in the small one.

That is the sample-size lesson in its useful form. What you can detect is the effect
*divided by* the spread of the thing you are measuring, and the spread is a property of your
data you can read off the ordinary days before you look at the event. A student who fits the
ordinary Thursdays first knows in advance whether ten release days can settle the question.

## 4.5 Your turn — before and after, on your own chat

`load_own_chat()` raises when there is no export yet — a your-turn section without data has
nothing to test. If the next cell errors, run [01.3](../lesson1/01.3-your-own-chat.ipynb)
first and point `current` in `config.toml` at the file it writes.

In [ ]:
own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])

### Think in families first

Before fitting anything, decide what kind of variable you are looking at and what a change in
it would mean — the family tells you which parameter to read:

| variable in your chat | family | what a change means |
|---|---|---|
| messages per day | Poisson if the rate is steady; **nbinom** when it wanders (it will) | the **rate** moved, or the **dispersion** did — busier, or more erratic |
| seconds between messages, within a burst | **exponential** | the **scale** (mean gap) moved — faster or slower back-and-forth |
| messages per day per person | Poisson / nbinom, one per author | *who* changed, not just how much |

A single fit over your whole chat is rarely interesting: "messages per day is over-dispersed"
is true of every chat, because no group keeps a steady rate for years, and "message length
is lognormal" is true of every chat for the reason §4.3 gave. The interesting version is the
one §4.4 just did: **an event you know about splits the data, and the fit on each side says
what changed.** 03.3 asked you to name such an event; use it here.

In [ ]:
my_event = None  # >>> Your turn: "YYYY-MM-DD" of an event you KNOW about (from 03.3) <<<

if my_event is None:
    # Not an event, just a way to make the cells below run: the middle of the chat.
    # Replace it. A split you cannot name is a split you cannot interpret.
    my_event = str(own["timestamp"].median().date())
    print(f"no event set; splitting at the midpoint {my_event} so the code runs")

per_day = (
    own.set_index("timestamp").resample("D").size().rename("messages").reset_index()
)
per_day["after"] = per_day["timestamp"] >= pd.Timestamp(my_event, tz=per_day["timestamp"].dt.tz)
print(per_day.groupby("after").messages.agg(["count", "mean", "var"]).round(1))

**Messages per day, both sides.** Fit the discrete families on each side. Read two things
from the parameters, not one: did the *mean* move, and did the *dispersion* — a group can
send the same number of messages in a much more erratic way, and that is a different finding.

In [ ]:
fitter = DistributionFitter(DistributionRegistry(), seed=42)
for label, side in [("before", per_day[~per_day.after]), ("after", per_day[per_day.after])]:
    fits = fitter.fit(side.messages.to_numpy().astype(float), discrete=True)
    best = next(f for f in fits if isinstance(f, FitResult) and f.best_likelihood)
    print(f"{label:7s} {best.distribution:8s} mean {best.frozen_dist.mean():6.1f}   "
          f"sd {best.frozen_dist.std():6.1f}   ({len(side)} days)")

In [ ]:
own_null = NullDistribution(
    lambda days: days.groupby("after").messages.mean().diff().iloc[-1], n_iter=2000, seed=4,
).run(per_day, label="after")

fig, ax = NullPlot(PlotSettings(figsize=(8, 4), title=f"Messages per day: after {my_event} minus before",
                                xlabel="after − before, messages per day")).plot(result=own_null)
_ = ax.legend()

**Gaps between messages, both sides.** The exponential is the family for waiting times at a
steady rate, and "within a burst" is the only place a chat has anything like a steady rate —
so the gaps are cut at an hour, and the two fits are compared on their *scale*, the mean gap.

⚠️ This one needs second-resolution timestamps. IRC logs `[HH:MM]`, and over half of
consecutive IRC messages land in the same minute, so the gap is zero by construction — a
measurement problem, not a modelling one. WhatsApp exports carry seconds; the check below
refuses to fit if yours does not.

In [ ]:
ts = own["timestamp"].sort_values()
on_the_minute = ((ts.dt.second == 0) & (ts.dt.microsecond == 0)).mean()
if on_the_minute > 0.3:
    print(f"{on_the_minute:.0%} of timestamps land exactly on the minute: too coarse for a gap fit.")
else:
    gaps = pd.DataFrame({"gap": ts.diff().dt.total_seconds(), "timestamp": ts}).dropna()
    gaps = gaps[(gaps.gap > 0) & (gaps.gap < 3600)]  # within an hour: a burst, not overnight
    gaps["after"] = gaps.timestamp >= pd.Timestamp(my_event, tz=gaps.timestamp.dt.tz)
    for label, side in [("before", gaps[~gaps.after]), ("after", gaps[gaps.after])]:
        fit = fitter.fit_distribution("exponential", side.gap.to_numpy())
        print(f"{label:7s} mean gap {fit.frozen_dist.mean():6.0f} s   ({len(side):,} gaps)")  # ty: ignore[unresolved-attribute]

### What to write down

1. **The event and what you expected it to change** — rate, dispersion, or gap scale — before
   you ran the cells. Quote yourself.
2. **The two fits**, as a sentence with parameters in it: "before, nbinom with mean 31 and sd
   19; after, mean 52 and sd 40". A table is fine; a screenshot of a histogram is not.
3. **The null plot, and where the real value sits in it.** If it sits inside the cloud, say so
   — a change you cannot distinguish from chance is a finding about your sample size, and
   §4.4 showed how to know that in advance from the spread of the ordinary days.

This is what the `goad` MCP's `goad_analysis_checklist` is built to interview you about; a
before/after question with a named event is exactly the shape it expects.

## Reflection

1. Pick one family from §4.1 and name a variable in your own chat you would expect to
   follow it. What would have to be true about *how that variable is generated* for the
   expectation to hold — and what would make the fit fail the way Poisson failed in §4.4?
2. In §4.3 the log-transform turned an "outlier" into an ordinary observation. Where in your
   own data would the reverse happen — a log making an ordinary point look suspicious?
3. §4.4 found the same lift with two different verdicts. For your own event: from the spread
   of the ordinary days alone, how many event days would you have needed to be sure?